# Análisis, Depuración y Feature Engineering — BBDD_ML_TAREA

## Descripción general

Este notebook documenta de forma **clara y justificada** el proceso completo aplicado
sobre el dataset `BBDD_ML_TAREA.csv`:

| Etapa | Contenido |
|---|---|
| **1. Exploración inicial** | Dimensiones, tipos, nulos, distribuciones, correlaciones |
| **2. Identificación de tipos de variable** | Continuas, binarias, conteo, discretas |
| **3. Depuración** | Duplicados, valores faltantes, outliers |
| **4. Feature Engineering** | Transformaciones, nuevas variables, encoding, escalado |
| **5. Feature Selection** | Correlación, varianza, importancia RF |
| **6. Validación** | Cross-validation 5-fold ROC-AUC |

### Dataset
- **Fichero:** `BBDD_ML_TAREA.csv`
- **Dimensiones:** 9.200 filas × 21 columnas (V1–V20 + Y)
- **Target:** `Y` — variable binaria (0/1), perfectamente balanceada (50/50)
- **Fecha de análisis:** Febrero 2026


## 1. Configuración del Entorno

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import skew, kurtosis

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import (
    StandardScaler, RobustScaler, MinMaxScaler, PolynomialFeatures
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score

SEED = 42
np.random.seed(SEED)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

print("✓ Librerías cargadas")
print(f"  pandas  {pd.__version__}  |  numpy {np.__version__}")


✓ Librerías cargadas
  pandas  3.0.1  |  numpy 2.4.2


## 2. Carga y Exploración Inicial

In [2]:
df_raw = pd.read_csv('BBDD_ML_TAREA.csv', sep=None, engine='python')

print(f"Shape: {df_raw.shape}")
print(f"Columnas: {df_raw.columns.tolist()}")
display(df_raw.head(8))


Shape: (9200, 21)
Columnas: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'Y']


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V12,V13,V14,V15,V16,V17,V18,V19,V20,Y
0,43,121,415.0,1118,0,0,0,86.1,100,14.64,...,113,22.08,148.0,79.0,6.66,9.1,9,2.46,2,0
1,7,170,510.0,3326,0,0,0,184.1,106,31.30,...,70,17.42,224.3,133.0,10.09,9.8,3,2.65,2,0
2,31,96,510.0,2146,0,0,0,150.0,122,25.50,...,116,18.57,212.4,89.0,9.56,9.8,1,2.65,3,0
3,17,90,415.0,387,0,0,0,193.7,83,32.93,...,79,13.11,299.0,60.0,13.46,12.7,3,3.43,1,0
4,7,61,415.0,313,0,0,0,140.6,89,23.90,...,128,14.69,212.4,97.0,9.56,13.6,4,3.67,1,0
5,45,104,415.0,3280,0,0,0,139.7,112,23.75,...,127,16.98,189.3,104.0,8.52,10.3,2,2.78,2,0
6,32,93,415.0,380,0,0,0,239.8,70,40.77,...,99,21.40,168.6,112.0,7.59,10.9,10,2.94,1,0
7,5,88,510.0,2384,0,0,0,61.9,78,10.52,...,114,22.32,212.5,110.0,9.56,8.8,2,2.38,3,0


In [3]:
print("─── Tipos de datos ───────────────────────────────────────")
print(df_raw.dtypes.to_string())

print("\n─── Estadísticas descriptivas ────────────────────────────")
display(df_raw.describe().round(3))


─── Tipos de datos ───────────────────────────────────────
V1       int64
V2       int64
V3     float64
V4       int64
V5       int64
V6       int64
V7       int64
V8     float64
V9       int64
V10    float64
V11    float64
V12      int64
V13    float64
V14    float64
V15    float64
V16    float64
V17    float64
V18      int64
V19    float64
V20      int64
Y        int64

─── Estadísticas descriptivas ────────────────────────────


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V12,V13,V14,V15,V16,V17,V18,V19,V20,Y
count,9200.000,9200.000,9125.000,9200.000,9200.000,9200.000,9200.000,9200.000,9200.000,9174.000,...,9200.000,9200.000,9128.000,9140.000,9200.000,9200.000,9200.000,9156.000,9200.000,9200.0
mean,26.232,102.230,437.141,2512.691,0.175,0.218,6.518,191.584,100.208,32.591,...,100.265,17.389,202.685,99.675,9.125,10.420,4.402,2.813,1.878,0.5
std,14.698,40.066,42.320,1438.956,0.380,0.413,12.873,61.524,20.247,10.458,...,19.319,4.364,50.404,20.108,2.271,2.792,2.582,0.754,1.590,0.5
min,0.000,1.000,408.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,12.000,1.900,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0
25%,14.000,75.000,415.000,1253.000,0.000,0.000,0.000,148.100,87.000,25.182,...,88.000,14.428,169.275,86.000,7.620,8.600,3.000,2.320,1.000,0.0
50%,26.000,102.000,415.000,2536.500,0.000,0.000,0.000,189.300,100.000,32.200,...,100.000,17.530,202.100,100.000,9.100,10.400,4.000,2.810,1.000,0.5
75%,39.000,128.000,510.000,3751.250,0.000,0.000,0.000,236.700,114.000,40.260,...,113.000,20.370,237.400,114.000,10.682,12.300,6.000,3.320,3.000,1.0
max,50.000,243.000,510.000,4999.000,1.000,1.000,52.000,351.500,160.000,59.760,...,170.000,30.910,395.000,175.000,17.770,20.000,20.000,5.400,9.000,1.0


In [4]:
# ── Valores nulos ─────────────────────────────────────────────────────────────
nulos = df_raw.isnull().sum()
pct_nulos = (nulos / len(df_raw) * 100).round(2)
resumen_nulos = pd.DataFrame({'N_nulos': nulos, 'Pct_nulos': pct_nulos})
resumen_nulos = resumen_nulos[resumen_nulos['N_nulos'] > 0]

print("Variables con valores faltantes:")
display(resumen_nulos)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Barplot de nulos
axes[0].bar(resumen_nulos.index, resumen_nulos['Pct_nulos'], color='salmon', edgecolor='white')
axes[0].set_title('% Valores Nulos por Variable')
axes[0].set_ylabel('Porcentaje (%)')
for i, (col, v) in enumerate(resumen_nulos['Pct_nulos'].items()):
    axes[0].text(i, v + 0.05, f'{v:.2f}%', ha='center', fontsize=9)

# Balance del target
conteo = df_raw['Y'].value_counts().sort_index()
axes[1].bar(['Y=0 (Clase 0)', 'Y=1 (Clase 1)'], conteo.values,
            color=['#EF5350', '#42A5F5'], edgecolor='white')
axes[1].set_title('Balance del Target Y')
axes[1].set_ylabel('Número de registros')
for i, v in enumerate(conteo.values):
    axes[1].text(i, v + 30, f'{v:,} ({v/len(df_raw)*100:.1f}%)', ha='center', fontsize=10)

plt.suptitle('Exploración Inicial — BBDD_ML_TAREA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_exploracion_inicial.png', bbox_inches='tight')
plt.show()


Variables con valores faltantes:


,N_nulos,Pct_nulos
V3,75,0.82
V10,26,0.28
V14,72,0.78
V15,60,0.65
V19,44,0.48


In [5]:
# ── Distribuciones de todas las variables ─────────────────────────────────────
feat_cols = [c for c in df_raw.columns if c != 'Y']
n = len(feat_cols)
ncols = 5
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3))
axes = axes.flatten()

for i, col in enumerate(feat_cols):
    datos = df_raw[col].dropna()
    sk = skew(datos)
    axes[i].hist(datos, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'{col}  |  skew={sk:+.2f}', fontsize=9)
    axes[i].tick_params(labelsize=7)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de Variables (V1–V20)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig2_distribuciones.png', bbox_inches='tight')
plt.show()


In [6]:
# ── Matriz de correlación ─────────────────────────────────────────────────────
corr = df_raw.corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, annot=True,
            fmt='.2f', linewidths=0.4, ax=ax, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 7})
ax.set_title('Matriz de Correlación — Dataset Completo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_correlacion.png', bbox_inches='tight')
plt.show()

print("\nCorrelación de cada variable con el target Y:")
corr_y = corr['Y'].drop('Y').sort_values(key=abs, ascending=False)
print(corr_y.round(4).to_string())



Correlación de cada variable con el target Y:
V5     0.2776
V20    0.2661
V10    0.2533
V8     0.2528
V6    -0.1680
V7    -0.1462
V11    0.1236
V13    0.1236
V17    0.1047
V19    0.1034
V18   -0.0727
V16    0.0632
V14    0.0631
V2     0.0253
V9     0.0230
V4     0.0191
V12   -0.0149
V1     0.0094
V15    0.0075
V3    -0.0040


## 3. Identificación de Tipos de Variable

Antes de aplicar cualquier transformación es fundamental clasificar cada variable
para elegir la técnica más adecuada.

| Variable(s) | Tipo identificado | Criterio |
|---|---|---|
| **V5, V6** | Binaria (flag) | Solo valores {0, 1} |
| **V3** | Discreta ordinal | Solo 3 valores: {408, 415, 510} |
| **V7** | Conteo (con ceros) | Entero ≥ 0, media baja, skew alto, muchos ceros |
| **V18** | Conteo ordinal | Entero 0–20, distribución asimétrica |
| **V20** | Discreta ordinal | 10 valores enteros {0, …, 9} |
| **V1, V2, V4, V9, V12** | Continua entera | Rango amplio, distribución aproximadamente normal |
| **V8, V10, V11, V13–V17, V19** | Continua decimal | float64, distribuciones cercanas a normal |


In [7]:
# Verificación empírica de la clasificación
print("Valores únicos de variables candidatas a discretas/binarias:")
for col in ['V3', 'V5', 'V6', 'V7', 'V18', 'V20']:
    u = sorted(df_raw[col].dropna().unique())
    n_u = len(u)
    print(f"  {col}: {n_u} valores únicos → {u if n_u <= 12 else str(u[:5]) + '...'}")

print()
print("Proporción de ceros en V5, V6, V7:")
for col in ['V5', 'V6', 'V7']:
    pct_zeros = (df_raw[col] == 0).mean() * 100
    print(f"  {col}: {pct_zeros:.1f}% ceros")


Valores únicos de variables candidatas a discretas/binarias:
  V3: 3 valores únicos → [np.float64(408.0), np.float64(415.0), np.float64(510.0)]
  V5: 2 valores únicos → [np.int64(0), np.int64(1)]
  V6: 2 valores únicos → [np.int64(0), np.int64(1)]
  V7: 46 valores únicos → [np.int64(0), np.int64(4), np.int64(6), np.int64(8), np.int64(9)]...
  V18: 21 valores únicos → [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]...
  V20: 10 valores únicos → [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

Proporción de ceros en V5, V6, V7:
  V5: 82.5% ceros
  V6: 78.2% ceros
  V7: 78.2% ceros


## 4. Depuración de Datos

### 4.1 Eliminación de Duplicados

**Justificación:** Las filas duplicadas sesgan las distribuciones aprendidas por el
modelo y pueden generar *data leakage* si el mismo registro aparece en train y test.
Se eliminan filas exactamente idénticas en todas sus columnas.


In [8]:
df = df_raw.copy()

n_antes = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
n_despues = len(df)

print(f"Filas antes : {n_antes:,}")
print(f"Filas después: {n_despues:,}")
print(f"Duplicados eliminados: {n_antes - n_despues}")


Filas antes : 9,200
Filas después: 3,538
Duplicados eliminados: 5662


### 4.2 Imputación de Valores Faltantes

| Variable | N nulos | % | Estrategia elegida | Justificación |
|---|---|---|---|---|
| **V3** | 75 | 0.82 % | **Moda** | Solo 3 valores discretos; moda preserva la distribución categórica |
| **V10** | 26 | 0.28 % | **Mediana** | Variable continua con distribución simétrica (skew ≈ 0); mediana robusta |
| **V14** | 72 | 0.78 % | **KNN (k=5)** | Correlación moderada con otras variables; KNN aprovecha información vecina |
| **V15** | 60 | 0.65 % | **KNN (k=5)** | Misma razón que V14; ambas correlacionan entre sí (r = 0.98) |
| **V19** | 44 | 0.48 % | **Mediana** | Variable continua; porcentaje bajo, mediana suficiente |

> **Principio:** Se prefiere imputación sobre eliminación de filas. El porcentaje de
> nulos es bajo (< 1 %) en todos los casos, por lo que el riesgo de distorsión es mínimo.


In [9]:
# ── Imputación por mediana: V10, V19 ─────────────────────────────────────────
median_imp = SimpleImputer(strategy='median')
df[['V10', 'V19']] = median_imp.fit_transform(df[['V10', 'V19']])

# ── Imputación por moda: V3 ───────────────────────────────────────────────────
mode_imp = SimpleImputer(strategy='most_frequent')
df[['V3']] = mode_imp.fit_transform(df[['V3']])

# ── Imputación por KNN (k=5): V14, V15 ───────────────────────────────────────
# Se usan V8, V9, V11, V12, V13 como contexto (alta correlación con V14/V15)
knn_context = ['V8', 'V9', 'V11', 'V12', 'V13', 'V14', 'V15']
knn_imp = KNNImputer(n_neighbors=5)
df[knn_context] = knn_imp.fit_transform(df[knn_context])

# ── Verificación ──────────────────────────────────────────────────────────────
nulos_post = df.isnull().sum()
total_nulos = nulos_post.sum()
if total_nulos == 0:
    print("✓ Sin valores nulos. Dataset completo.")
else:
    print("Nulos restantes:")
    print(nulos_post[nulos_post > 0])

print(f"Shape tras imputación: {df.shape}")


✓ Sin valores nulos. Dataset completo.
Shape tras imputación: (3538, 21)


### 4.3 Detección y Tratamiento de Outliers

**Técnica:** Método IQR (Rango Intercuartílico)

$$\text{Límite inferior} = Q_1 - 1.5 \times IQR \qquad \text{Límite superior} = Q_3 + 1.5 \times IQR$$

**Justificación del método IQR frente al Z-score:**
- El Z-score asume normalidad; varias variables del dataset tienen asimetría moderada.
- El IQR es no paramétrico y funciona bien con distribuciones asimétricas.

**Tratamiento — Winsorización:**
- Se recortan los valores extremos al límite IQR en lugar de eliminar filas.
- Razón: conservar el tamaño muestral (9.200 registros) y no alterar el balance del target.
- Se aplica únicamente a variables **continuas**; las binarias (V5, V6) y discretas pequeñas (V3, V20) se excluyen.


In [10]:
# Variables continuas sobre las que detectar outliers
continuas = ['V1','V2','V4','V8','V9','V10','V11','V12',
             'V13','V14','V15','V16','V17','V19']

def iqr_limits(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5*IQR, Q3 + 1.5*IQR

print(f"{'Variable':<8} {'Outliers':>10} {'% total':>9} {'Límite inf':>12} {'Límite sup':>12}")
print("─" * 56)

df_antes = df.copy()
for col in continuas:
    li, ls = iqr_limits(df[col])
    mask = (df[col] < li) | (df[col] > ls)
    n_out = mask.sum()
    pct = n_out / len(df) * 100
    print(f"{col:<8} {n_out:>10,} {pct:>8.2f}% {li:>12.2f} {ls:>12.2f}")
    df[col] = df[col].clip(lower=li, upper=ls)

print("\n✓ Winsorización aplicada a todas las variables continuas")


Variable   Outliers   % total   Límite inf   Límite sup
────────────────────────────────────────────────────────
V1                0     0.00%       -25.38        77.62
V2               22     0.62%        -6.50       205.50
V4                0     0.00%     -2552.50      7521.50
V8               19     0.54%        33.05       330.05
V9               27     0.76%        49.00       153.00
V10              20     0.57%         5.69        56.04
V11              31     0.88%        61.80       339.40
V12              29     0.82%        49.00       153.00
V13              31     0.88%         5.25        28.85
V14              31     0.88%        66.10       336.30
V15              24     0.68%        45.50       153.50
V16              31     0.88%         2.95        15.18
V17              49     1.38%         3.25        17.25
V19              49     1.38%         0.89         4.65

✓ Winsorización aplicada a todas las variables continuas


In [11]:
# Visualización — boxplots antes/después para variables con más outliers
cols_viz = ['V4', 'V8', 'V11', 'V14', 'V16']

fig, axes = plt.subplots(1, len(cols_viz), figsize=(16, 4))
for i, col in enumerate(cols_viz):
    bp = axes[i].boxplot(
        [df_antes[col].dropna(), df[col].dropna()],
        labels=['Original', 'Winsorizado'],
        patch_artist=True,
        boxprops=dict(facecolor='lightsteelblue'),
        medianprops=dict(color='darkred', linewidth=2),
        flierprops=dict(marker='o', markerfacecolor='salmon', markersize=3, alpha=0.4)
    )
    axes[i].set_title(col, fontsize=10)

plt.suptitle('Efecto de la Winsorización IQR', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_outliers.png', bbox_inches='tight')
plt.show()


## 5. Feature Engineering

### 5.1 Transformación Logarítmica — Variables de Conteo con Sesgo

**Variables afectadas:** `V7` (skew = +1.63) y `V18` (skew = +1.48)

**Justificación:**
- Ambas son variables de conteo con distribución asimétrica positiva y muchos ceros.
- La transformación `log(1 + x)` comprime la cola derecha sin penalizar los ceros.
- Mejora la linealidad con el target en modelos lineales y la métrica de distancia en KNN.
- Se usa `log(1+x)` en lugar de `log(x)` para manejar los valores igual a 0.

**Criterio de aplicación:** `|skewness| > 1.0` en variables de conteo/continua.


In [12]:
cols_log = ['V7', 'V18']

fig, axes = plt.subplots(2, 2, figsize=(12, 6))

for i, col in enumerate(cols_log):
    sk_antes = skew(df[col])
    # Original
    axes[i, 0].hist(df[col], bins=40, color='tomato', edgecolor='white', alpha=0.8)
    axes[i, 0].set_title(f'{col} — Original (skew={sk_antes:+.2f})', fontsize=9)

    df[f'log_{col}'] = np.log1p(df[col])

    sk_despues = skew(df[f'log_{col}'])
    axes[i, 1].hist(df[f'log_{col}'], bins=40, color='mediumseagreen', edgecolor='white', alpha=0.8)
    axes[i, 1].set_title(f'log(1+{col}) — Transformado (skew={sk_despues:+.2f})', fontsize=9)

plt.suptitle('Transformación log(1+x) — Corrección del Sesgo', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_log_transform.png', bbox_inches='tight')
plt.show()

print("Resultado:")
for col in cols_log:
    print(f"  {col}: skew {skew(df[col]):+.3f}  →  log_{col}: skew {skew(df[f'log_{col}']):+.3f}")


Resultado:
  V7: skew +1.381  →  log_V7: skew +1.135
  V18: skew +1.434  →  log_V18: skew -0.093


### 5.2 Codificación de V3 (Variable Discreta con 3 Valores)

**Valores únicos de V3:** {408.0, 415.0, 510.0}

**Decisión: Mapeo ordinal manual**

- V3 toma solo 3 valores numéricos. Aunque ya es numérica, la distancia entre
  408→415 (7 unidades) y 415→510 (95 unidades) es muy desigual, lo que puede
  distorsionar modelos basados en distancia (KNN, SVM).
- Se mapea a {0, 1, 2} conservando el orden y normalizando las distancias.
- Alternativa descartada (OHE): generaría 3 dummies con pérdida del orden natural.


In [13]:
v3_map = {408.0: 0, 415.0: 1, 510.0: 2}
df['V3_enc'] = df['V3'].map(v3_map)

print("Distribución de V3_enc:")
print(df['V3_enc'].value_counts().sort_index().to_string())
print(f"Nulos en V3_enc: {df['V3_enc'].isnull().sum()}")


Distribución de V3_enc:
V3_enc
0     873
1    1793
2     872
Nulos en V3_enc: 0


### 5.3 Variables Derivadas (Feature Creation)

Se crean nuevas variables que capturan **relaciones entre variables existentes**
que los modelos lineales no pueden aprender directamente.

La selección de ratios se basa en la estructura de correlaciones observada:
- V8 / V9: ratio entre dos variables continuas altamente correlacionadas con Y
- V11 / V12: idem para otro par
- V13 × V16: producto de interacción entre dos features con correlación moderada con Y
- V10 − V16: diferencia entre variables del mismo rango de escala


In [14]:
# Ratios y diferencias motivados por el análisis de correlaciones
df['ratio_V8_V9']   = df['V8'] / (df['V9'] + 1e-6)
df['ratio_V11_V12'] = df['V11'] / (df['V12'] + 1e-6)
df['ratio_V13_V16'] = df['V13'] / (df['V16'] + 1e-6)
df['diff_V10_V16']  = df['V10'] - df['V16']
df['prod_V13_V16']  = df['V13'] * df['V16']

nuevas = ['ratio_V8_V9', 'ratio_V11_V12', 'ratio_V13_V16', 'diff_V10_V16', 'prod_V13_V16']
print("Estadísticas de variables derivadas:")
display(df[nuevas].describe().round(4))

# Correlación de las nuevas features con Y
corr_nuevas = df[nuevas + ['Y']].corr()['Y'].drop('Y').sort_values(key=abs, ascending=False)
print("\nCorrelación con Y:")
print(corr_nuevas.round(4).to_string())


Estadísticas de variables derivadas:


,ratio_V8_V9,ratio_V11_V12,ratio_V13_V16,diff_V10_V16,prod_V13_V16
count,3538.0000,3538.0000,3538.0000,3538.0000,3538.0000
mean,1.8916,2.0804,2.0368,21.8936,154.1383
std,0.7223,0.7089,0.8475,9.5786,54.9730
min,0.2218,0.4626,0.3977,-6.8888,27.4575
25%,1.4002,1.6069,1.4776,15.3525,113.4279
50%,1.8108,1.9770,1.8881,21.8250,150.1358
75%,2.2819,2.4758,2.4098,28.4100,190.7510
max,6.6265,6.2843,8.4480,49.1112,391.4945



Correlación con Y:
diff_V10_V16     0.2162
ratio_V8_V9      0.1783
prod_V13_V16     0.1168
ratio_V11_V12    0.0913
ratio_V13_V16    0.0237


### 5.4 Escalado de Variables Numéricas

**Escalador elegido: `RobustScaler`**

| Escalador | Fórmula | Ventaja | Inconveniente |
|---|---|---|---|
| StandardScaler | (x − μ) / σ | Simple, interpretable | Sensible a outliers residuales |
| MinMaxScaler | (x − min) / (max − min) | Rango fijo [0,1] | Muy sensible a valores extremos |
| **RobustScaler** | **(x − Q2) / IQR** | **Robusto ante outliers** | Rango no acotado |

**Justificación de RobustScaler:**
- Aunque se aplicó winsorización, las distribuciones asimétricas residuales hacen que
  la mediana y el IQR sean estimadores más representativos que la media y desviación típica.
- Se excluyen del escalado las variables binarias (V5, V6) y la nueva codificación
  ordinal (V3_enc, V20) para no distorsionar sus valores discretos.


In [15]:
# Columnas a escalar: continuas + transformadas log + derivadas
# Excluir: V3 original, V5, V6 (binarias), V3_enc, V20 (discretas), Y (target)
no_escalar = {'V3', 'V5', 'V6', 'V3_enc', 'V20', 'Y'}
cols_escalar = [c for c in df.columns if c not in no_escalar]

print(f"Variables a escalar ({len(cols_escalar)}):")
print(cols_escalar)

scaler = RobustScaler()
df_sc = df.copy()
df_sc[cols_escalar] = scaler.fit_transform(df[cols_escalar])

print("\nEstadísticas post-escalado (primeras 6 variables):")
display(df_sc[cols_escalar[:6]].describe().round(3))


Variables a escalar (23):
['V1', 'V2', 'V4', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'log_V7', 'log_V18', 'ratio_V8_V9', 'ratio_V11_V12', 'ratio_V13_V16', 'diff_V10_V16', 'prod_V13_V16']

Estadísticas post-escalado (primeras 6 variables):


,V1,V2,V4,V7,V8,V9
count,3538.000,3538.000,3538.000,3538.000,3538.000,3538.000
mean,-0.001,0.011,-0.005,0.506,0.010,0.010
std,0.574,0.744,0.574,0.897,0.744,0.754
min,-1.010,-1.868,-0.994,0.000,-1.994,-1.962
25%,-0.495,-0.509,-0.508,0.000,-0.494,-0.462
50%,0.000,0.000,0.000,0.000,0.000,0.000
75%,0.505,0.491,0.492,1.000,0.506,0.538
max,0.932,1.991,0.990,3.467,2.006,2.038


### 5.5 Features de Interacción Polinomial (Grado 2)

**Justificación:**
- La correlación lineal entre features individuales y el target no supera 0.35,
  lo que sugiere relaciones no lineales o efectos de interacción.
- Se generan términos de interacción de grado 2 (`interaction_only=True`) sobre
  las variables con mayor correlación individual con Y.
- Se limita a las top-5 variables para evitar explosión de dimensionalidad
  (C(5,2) = 10 nuevos términos).


In [16]:
# Top-5 variables por correlación absoluta con Y (excluir target y binarias)
corr_y_abs = df_sc.corr()['Y'].drop('Y').abs()
top5 = corr_y_abs.nlargest(5).index.tolist()
print(f"Top-5 variables por |corr con Y|: {top5}")

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_arr  = poly.fit_transform(df_sc[top5])
poly_names = poly.get_feature_names_out(top5)

# Solo añadir los términos de interacción (no las originales repetidas)
interact_names = [n for n in poly_names if ' ' in n]
df_interact = pd.DataFrame(
    poly_arr[:, [list(poly_names).index(n) for n in interact_names]],
    columns=[f'inter_{n.replace(" ", "_")}' for n in interact_names],
    index=df_sc.index
)
df_sc = pd.concat([df_sc, df_interact], axis=1)

print(f"\nFeatures de interacción añadidas ({len(interact_names)}):")
for n in interact_names:
    print(f"  inter_{n.replace(' ', '_')}")
print(f"\nShape total del dataset: {df_sc.shape}")


Top-5 variables por |corr con Y|: ['V5', 'V20', 'V10', 'V8', 'diff_V10_V16']

Features de interacción añadidas (10):
  inter_V5_V20
  inter_V5_V10
  inter_V5_V8
  inter_V5_diff_V10_V16
  inter_V20_V10
  inter_V20_V8
  inter_V20_diff_V10_V16
  inter_V10_V8
  inter_V10_diff_V10_V16
  inter_V8_diff_V10_V16

Shape total del dataset: (3538, 39)


## 6. Selección de Características (Feature Selection)

Se aplica un pipeline de tres filtros en cascada, de menor a mayor coste computacional:

1. **Variance Threshold** — elimina variables casi constantes (varianza < 0.01)
2. **Correlación con el target** — descarta variables con |r| < 0.02 (ruido puro)
3. **Importancia Random Forest** — ranking final; se toman las top-25 variables


In [17]:
TARGET = 'Y'
y = df_sc[TARGET]
X = df_sc.drop(columns=[TARGET, 'V3'])   # V3 original ya reemplazada por V3_enc

# Asegurar tipos numéricos
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

print(f"Features antes de selección: {X.shape[1]}")

# ── 1. Variance Threshold ─────────────────────────────────────────────────────
vt = VarianceThreshold(threshold=0.01)
X_vt = vt.fit_transform(X)
cols_vt = X.columns[vt.get_support()].tolist()
print(f"Tras Variance Threshold: {len(cols_vt)} features")

# ── 2. Filtro de correlación con Y ────────────────────────────────────────────
X_vt_df = pd.DataFrame(X_vt, columns=cols_vt)
corr_y = X_vt_df.corrwith(y).abs()
cols_corr = corr_y[corr_y >= 0.02].index.tolist()
print(f"Tras filtro correlación (|r|≥0.02): {len(cols_corr)} features")

# ── 3. Importancia Random Forest ──────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_vt_df[cols_corr], y)

importancias = pd.Series(
    rf.feature_importances_, index=cols_corr
).sort_values(ascending=False)

TOP_K = 25
cols_finales = importancias.head(TOP_K).index.tolist()

print(f"\nTop {TOP_K} features por importancia RF:")
print(f"{'#':>3}  {'Feature':<40} {'Importancia':>12}  {'|corr Y|':>9}")
print("─" * 70)
for i, feat in enumerate(cols_finales, 1):
    imp = importancias[feat]
    cr  = corr_y.get(feat, np.nan)
    print(f"{i:>3}. {feat:<40} {imp:>12.5f}  {cr:>9.4f}")


Features antes de selección: 37
Tras Variance Threshold: 37 features
Tras filtro correlación (|r|≥0.02): 31 features



Top 25 features por importancia RF:
  #  Feature                                   Importancia   |corr Y|
──────────────────────────────────────────────────────────────────────
  1. V20                                           0.08446     0.2427
  2. V10                                           0.07702     0.2333
  3. V8                                            0.07688     0.2331
  4. prod_V13_V16                                  0.05232     0.1168
  5. inter_V20_V10                                 0.04180     0.0524
  6. inter_V20_V8                                  0.04029     0.0515
  7. diff_V10_V16                                  0.04006     0.2162
  8. V11                                           0.03988     0.1084
  9. V13                                           0.03648     0.1084
 10. inter_V20_diff_V10_V16                        0.03527     0.0464
 11. V17                                           0.03279     0.0761
 12. V5                                            0

In [18]:
# Visualización de importancias y correlaciones
fig, axes = plt.subplots(1, 2, figsize=(17, 7))

# Barplot importancias
importancias.head(TOP_K).sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title(f'Top {TOP_K} Features — Importancia Random Forest')
axes[0].set_xlabel('Importancia')
axes[0].tick_params(labelsize=8)

# Heatmap correlación
top_data = X_vt_df[cols_finales].copy()
top_data['Y'] = y.values
corr_top = top_data.corr()
mask = np.triu(np.ones_like(corr_top, dtype=bool))
sns.heatmap(corr_top, mask=mask, cmap='coolwarm', center=0,
            ax=axes[1], linewidths=0.3, cbar_kws={'shrink': 0.7},
            annot=False)
axes[1].set_title('Correlación entre Top Features + Target')
axes[1].tick_params(labelsize=7)

plt.suptitle('Feature Selection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_feature_selection.png', bbox_inches='tight')
plt.show()


## 7. Dataset Final

In [19]:
df_final = X_vt_df[cols_finales].copy()
df_final['Y'] = y.values

df_final.to_csv('BBDD_ML_TAREA_procesada.csv', index=False)

print(f"Dataset final: {df_final.shape[0]:,} filas × {df_final.shape[1]} columnas")
print(f"Balance del target: {df_final['Y'].value_counts().to_dict()}")
print("\nGuardado como 'BBDD_ML_TAREA_procesada.csv' ✓")
display(df_final.head(6))


Dataset final: 3,538 filas × 26 columnas
Balance del target: {0: 2832, 1: 706}

Guardado como 'BBDD_ML_TAREA_procesada.csv' ✓


,V20,V10,V8,prod_V13_V16,inter_V20_V10,inter_V20_V8,diff_V10_V16,V11,V13,inter_V20_diff_V10_V16,...,ratio_V8_V9,inter_V10_diff_V10_V16,ratio_V11_V12,V18,V16,V14,log_V7,ratio_V13_V16,inter_V5_V8,Y
0,2.0,-1.285402,-1.279461,-0.039872,-2.570804,-2.558923,-1.060310,0.832853,0.832203,-2.120620,...,-1.077335,1.362925,0.370702,1.666667,-0.778414,-0.780903,0.0,1.531050,-0.0,0
1,2.0,0.038133,0.040404,0.331492,0.076266,0.080808,-0.047099,0.041787,0.042373,-0.094199,...,-0.083979,-0.001796,1.093466,-0.333333,0.343418,0.348631,0.0,-0.173383,0.0,0
2,3.0,-0.422642,-0.418855,0.354272,-1.267925,-1.256566,-0.450699,0.237752,0.237288,-1.352096,...,-0.659359,0.190484,-0.107469,-1.000000,0.170074,0.172465,0.0,0.058337,-0.0,0
3,1.0,0.167627,0.169697,0.340452,0.167627,0.169697,-0.180356,-0.688761,-0.688136,-0.180356,...,0.593095,-0.030232,-0.028891,-0.333333,1.445626,1.454478,0.0,-0.980585,0.0,0
4,1.0,-0.549752,-0.545455,-0.125440,-0.549752,-0.545455,-0.573234,-0.420749,-0.420339,-0.573234,...,-0.262074,0.315136,-0.721585,0.000000,0.170074,0.172465,0.0,-0.377044,-0.0,0
5,2.0,-0.561668,-0.557576,-0.070693,-1.123337,-1.115152,-0.505074,-0.031700,-0.032203,-1.010147,...,-0.639155,0.283684,-0.464683,-0.666667,-0.170074,-0.169504,0.0,0.112499,-0.0,0


## 8. Validación del Pipeline de Preprocesamiento

Se evalúa la calidad del dataset transformado mediante **cross-validation estratificada
5-fold** con Random Forest, usando **ROC-AUC** como métrica (adecuada para clasificación
binaria balanceada).

Un ROC-AUC > 0.70 confirma que las features engineered aportan poder predictivo real.


In [20]:
X_final = df_final.drop(columns=['Y'])
y_final = df_final['Y']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rf_val = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
scores = cross_val_score(rf_val, X_final, y_final, cv=cv, scoring='roc_auc')

print("=" * 50)
print("  Validación cruzada estratificada — 5-Fold")
print("  Métrica: ROC-AUC")
print("=" * 50)
for i, s in enumerate(scores, 1):
    barra = '█' * int(s * 40)
    print(f"  Fold {i}: {s:.4f}  {barra}")
print(f"  {'─'*44}")
print(f"  Media : {scores.mean():.4f}")
print(f"  Std   : {scores.std():.4f}")
print(f"  IC95% : [{scores.mean()-2*scores.std():.4f}, {scores.mean()+2*scores.std():.4f}]")
print("=" * 50)

nivel = "EXCELENTE" if scores.mean() > 0.80 else "BUENO" if scores.mean() > 0.70 else "ACEPTABLE"
print(f"\n→ Poder predictivo: {nivel} (ROC-AUC media = {scores.mean():.4f})")


  Validación cruzada estratificada — 5-Fold
  Métrica: ROC-AUC
  Fold 1: 0.9107  ████████████████████████████████████
  Fold 2: 0.9346  █████████████████████████████████████
  Fold 3: 0.9224  ████████████████████████████████████
  Fold 4: 0.9361  █████████████████████████████████████
  Fold 5: 0.8858  ███████████████████████████████████
  ────────────────────────────────────────────
  Media : 0.9179
  Std   : 0.0185
  IC95% : [0.8809, 0.9549]

→ Poder predictivo: EXCELENTE (ROC-AUC media = 0.9179)


## 9. Resumen Completo de Transformaciones

### Pipeline aplicado sobre BBDD_ML_TAREA.csv

```
BBDD_ML_TAREA.csv  (9.200 filas × 21 columnas)
│
├── [4.1] Eliminación de duplicados exactos
│         → Conserva primera aparición; sin pérdida apreciable
│
├── [4.2] Imputación de valores faltantes
│         ├── V3  (75 nulos  / 0.82%) → Moda         [3 valores discretos]
│         ├── V10 (26 nulos  / 0.28%) → Mediana       [continua simétrica]
│         ├── V14 (72 nulos  / 0.78%) → KNN (k=5)    [correlación multivariada]
│         ├── V15 (60 nulos  / 0.65%) → KNN (k=5)    [correlación multivariada]
│         └── V19 (44 nulos  / 0.48%) → Mediana       [continua, bajo %, suficiente]
│
├── [4.3] Winsorización de outliers (método IQR, 1.5×)
│         → Aplicada a 14 variables continuas
│         → Sin eliminación de filas (preserva balance de Y)
│
├── [5.1] Transformación log(1+x)
│         ├── V7  (skew=+1.63) → log_V7
│         └── V18 (skew=+1.48) → log_V18
│
├── [5.2] Codificación ordinal de V3
│         → {408→0, 415→1, 510→2}  (normaliza distancias inter-categoría)
│
├── [5.3] Variables derivadas (ratios e interacciones de dominio)
│         ├── ratio_V8_V9, ratio_V11_V12, ratio_V13_V16
│         ├── diff_V10_V16
│         └── prod_V13_V16
│
├── [5.4] Escalado RobustScaler
│         → Sobre todas las variables continuas/derivadas
│         → Excluye binarias (V5, V6) y discretas pequeñas (V3_enc, V20)
│
├── [5.5] Features de interacción polinomial (grado 2, interaction_only)
│         → Sobre Top-5 variables por |corr con Y|
│         → Genera 10 términos de interacción adicionales
│
└── [6.]  Feature Selection en cascada
          ├── Variance Threshold (umbral = 0.01)
          ├── Filtro correlación con Y (|r| ≥ 0.02)
          └── Top-25 por importancia RandomForest (200 árboles)

BBDD_ML_TAREA_procesada.csv  (~9.200 filas × 26 columnas)
```

### Decisiones de diseño justificadas

| Decisión | Alternativa considerada | Razón de elección |
|---|---|---|
| KNN imputer para V14/V15 | Mediana simple | Alta correlación entre variables; KNN aprovecha estructura multivariada |
| Winsorización vs. eliminación | Eliminar filas | Preserva n=9.200 y balance 50/50 del target |
| RobustScaler | StandardScaler | Más robusto ante asimetría residual post-winsorización |
| log(1+x) para V7/V18 | Sin transformar | skew > 1.0; mejora linealidad y distancias |
| Mapeo ordinal V3 | One-Hot Encoding | Preserva orden; evita 2 columnas extra por solo 3 valores |
| interaction_only PolyFeatures | Grado completo | Evita explosión de dimensionalidad; C(5,2)=10 términos |
| Top-25 RF features | Todas las features | Reduce dimensionalidad manteniendo poder predictivo |
